In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Literal
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
import operator
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_groq import ChatGroq

from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [5]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [7]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)
model = ChatHuggingFace(llm=llm)


In [8]:
def chat_node(state:ChatState):
    messages = state['messages']
    response =model.invoke(messages)
    return {'messages':[response]}

In [17]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
# chatbot = graph.compile()
chatbot = graph.compile(checkpointer=checkpointer)

In [15]:
initial_state = {
    'messages':[HumanMessage(content='What is the capital of India')]
}
chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is New Delhi.'

In [18]:
thread_id = '1'

while True:
    user_message = input('Type here: ')
    print('User: ',user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI: ',response['messages'][-1].content)

User:  hi
AI:  How's it going? Is there something I can help you with or would you like to chat?
User:  my name is Ayyan
AI:  Nice to meet you, Ayyan! How's your day going so far?
User:  tell me what is my name
AI:  Your name is Ayyan.
User:  10+654
AI:  To calculate the sum, I'll add 10 and 654:

10 + 654 = 664
User:  +344
AI:  To calculate the sum, I'll add 664 and 344:

664 + 344 = 1008
User:  quit
